<a href="https://colab.research.google.com/github/natdanaiii/Trading/blob/main/Grid_trading.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BTC Spot Grid Trading Backtest

**Workflow:** Setup → Historical Data → Excel Grid Model → Historical Grid Configuration → Backtest Engine → Results

## 1. Setup

In [ ]:
import os
import io
import glob
import datetime
import heapq
import bisect

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/03.Trading/00.Live Trading'
print(f'Data directory: {DATA_DIR}')

## 2. Configuration

In [ ]:
# Market data configuration
SYMBOL = 'BTCUSDT'
START_DATE = '2024-01-01'
END_DATE = '2026-01-01'
TIMEFRAMES = ['1m', '1h', '1d']

# KZM Excel-template parameters used only for replication check
GRID_CAPITAL = 3000.0
GRID_CEILING = 8987.0
GRID_FLOOR = 1987.0
GRID_GAP = 70.0

# Trading fees
BUY_FEE = 0.001
SELL_FEE = 0.001

## 3. Historical Data

The first backtest version uses **BTCUSDT 1-minute OHLCV** as the execution data.

In [ ]:
def download_binance_history(symbol, start_date_str, end_date_str, timeframes, data_dir):
    start_date = datetime.datetime.strptime(start_date_str, '%Y-%m-%d')
    end_date = datetime.datetime.strptime(end_date_str, '%Y-%m-%d')
    os.makedirs(data_dir, exist_ok=True)
    current_date = start_date

    while current_date < end_date:
        year, month = current_date.year, current_date.month
        for timeframe in timeframes:
            url = (
                'https://data.binance.vision/data/spot/monthly/klines/'
                f'{symbol}/{timeframe}/{symbol}-{timeframe}-{year}-{month:02d}.zip'
            )
            try:
                print(f'Downloading {symbol} {timeframe} for {year}-{month:02d}...')
                response = requests.get(url, timeout=60)
                response.raise_for_status()
                columns = [
                    'open_time', 'open', 'high', 'low', 'close', 'volume',
                    'close_time', 'quote_volume', 'number_of_trades',
                    'taker_buy_base', 'taker_buy_quote', 'ignore'
                ]
                df_month = pd.read_csv(io.BytesIO(response.content), compression='zip', header=None, names=columns)
                time_unit = 'us' if year >= 2025 else 'ms'
                df_month['open_time'] = pd.to_datetime(df_month['open_time'], unit=time_unit, utc=True)
                df_month['close_time'] = pd.to_datetime(df_month['close_time'], unit=time_unit, utc=True)
                numeric_cols = ['open', 'high', 'low', 'close', 'volume']
                df_month[numeric_cols] = df_month[numeric_cols].astype(float)
                save_path = os.path.join(data_dir, f'{symbol}-{timeframe}-{year}-{month:02d}.csv')
                df_month.to_csv(save_path, index=False)
                print(f'Saved: {save_path}')
            except Exception as exc:
                print(f'Could not download {symbol} {timeframe} {year}-{month:02d}: {exc}')

        if current_date.month == 12:
            current_date = current_date.replace(year=current_date.year + 1, month=1)
        else:
            current_date = current_date.replace(month=current_date.month + 1)

In [ ]:
def combine_monthly_csv(symbol, timeframes, data_dir):
    output_paths = {}
    for timeframe in timeframes:
        pattern = os.path.join(data_dir, f'{symbol}-{timeframe}-????-??.csv')
        file_list = sorted(glob.glob(pattern))
        if not file_list:
            print(f'No monthly files found for timeframe: {timeframe}')
            continue
        combined = pd.concat([pd.read_csv(path) for path in file_list], ignore_index=True)
        combined['open_time'] = pd.to_datetime(combined['open_time'], utc=True)
        combined = combined.drop_duplicates(subset='open_time').sort_values('open_time').reset_index(drop=True)
        output_path = os.path.join(data_dir, f'{symbol}-{timeframe}-combined.csv')
        combined.to_csv(output_path, index=False)
        output_paths[timeframe] = output_path
        print(f'Created: {output_path} ({len(combined):,} rows)')
    return output_paths

In [ ]:
def load_market_data(symbol, timeframe, data_dir):
    file_path = os.path.join(data_dir, f'{symbol}-{timeframe}-combined.csv')
    if not os.path.exists(file_path):
        raise FileNotFoundError(f'Combined data file not found: {file_path}')

    df = pd.read_csv(file_path)
    df['open_time'] = pd.to_datetime(df['open_time'], utc=True)
    numeric_cols = ['open', 'high', 'low', 'close', 'volume']
    df[numeric_cols] = df[numeric_cols].astype(float)
    return df.drop_duplicates(subset='open_time').sort_values('open_time').reset_index(drop=True)

# Primary execution dataset
df_1m = load_market_data(SYMBOL, '1m', DATA_DIR)

# Apply configured backtest date window
start_ts = pd.Timestamp(START_DATE, tz='UTC')
end_ts = pd.Timestamp(END_DATE, tz='UTC')
df_1m = df_1m.loc[(df_1m['open_time'] >= start_ts) & (df_1m['open_time'] < end_ts)].reset_index(drop=True)

df_1m.head()

In [ ]:
def validate_market_data(df):
    ohlc_cols = ['open', 'high', 'low', 'close']
    print(f'Rows              : {len(df):,}')
    print(f'Start             : {df["open_time"].min()}')
    print(f'End               : {df["open_time"].max()}')
    print(f'Duplicate times   : {df["open_time"].duplicated().sum():,}')
    print(f'Rows missing OHLC : {df[ohlc_cols].isna().any(axis=1).sum():,}')

validate_market_data(df_1m)

## 4. Excel Grid Model

This section is only a baseline check against the KZM Excel template before historical execution is added.

In [ ]:
def build_excel_grid_table(capital, ceiling, floor, gap, buy_fee=0.001, sell_fee=0.001):
    if capital <= 0:
        raise ValueError('capital must be greater than 0.')
    if ceiling <= floor:
        raise ValueError('ceiling must be greater than floor.')
    if gap <= 0:
        raise ValueError('gap must be greater than 0.')

    raw_levels = (ceiling - floor) / gap
    if not np.isclose(raw_levels, round(raw_levels)):
        raise ValueError('(ceiling - floor) must be exactly divisible by gap.')

    n_levels = int(round(raw_levels))
    capital_per_level = capital / n_levels
    buy_prices = ceiling - gap * np.arange(1, n_levels + 1)
    sell_prices = buy_prices + gap
    gross_base_amount = capital_per_level / buy_prices
    buy_fee_base = gross_base_amount * buy_fee
    base_amount = gross_base_amount - buy_fee_base
    gross_sell = base_amount * sell_prices
    sell_fee_quote = gross_sell * sell_fee
    net_sell = gross_sell - sell_fee_quote
    profit = net_sell - capital_per_level

    return pd.DataFrame({
        'level': np.arange(1, n_levels + 1),
        'buy_price': buy_prices,
        'sell_price': sell_prices,
        'capital_per_level': capital_per_level,
        'gross_base_amount': gross_base_amount,
        'buy_fee_base': buy_fee_base,
        'base_amount': base_amount,
        'gross_sell': gross_sell,
        'sell_fee_quote': sell_fee_quote,
        'net_sell': net_sell,
        'profit': profit
    })

df_grid_excel = build_excel_grid_table(GRID_CAPITAL, GRID_CEILING, GRID_FLOOR, GRID_GAP, BUY_FEE, SELL_FEE)
df_grid_excel.head()

In [ ]:
# Checkpoint: first Excel pair = 8,917 -> 8,987
excel_check = df_grid_excel.iloc[0]
print(f'Buy price  : {excel_check["buy_price"]:.2f}')
print(f'Sell price : {excel_check["sell_price"]:.2f}')
print(f'Base amount: {excel_check["base_amount"]:.9f}')
print(f'Profit     : {excel_check["profit"]:.6f} USDT')

EXPECTED_FIRST_PROFIT = 0.175064
assert np.isclose(excel_check['profit'], EXPECTED_FIRST_PROFIT, atol=1e-6), 'Excel replication checkpoint failed.'
print('Excel replication checkpoint: PASSED')

## 5. Historical Grid Configuration — V0

The backtest keeps the **same parameter logic as the KZM Excel template**:

- Capital, Ceiling, Floor, Gap, Buy Fee, and Sell Fee are configuration inputs.
- Number of grids is calculated as `(Ceiling - Floor) / Gap`.
- Capital per grid is calculated as `Capital / Number of Grids`.

For V0, the historical BTCUSDT range is used only to select practical Floor/Ceiling values. The **grid gap is an explicit strategy input**, not derived from the number of grids.

**Important:** using the full-period historical Low/High to choose Floor/Ceiling introduces look-ahead bias. V0 is for validating execution logic only.

In [ ]:
# Backtest strategy inputs (same structure as the Excel template)
BACKTEST_CAPITAL = 3000.0
BACKTEST_GAP = 1000.0
PRICE_ROUNDING = 1000.0

historical_low = df_1m['low'].min()
historical_high = df_1m['high'].max()

# Historical data is used only to select practical V0 boundaries
BACKTEST_FLOOR = np.floor(historical_low / PRICE_ROUNDING) * PRICE_ROUNDING
BACKTEST_CEILING = np.ceil(historical_high / PRICE_ROUNDING) * PRICE_ROUNDING

# Excel logic: number of grids comes from Ceiling, Floor and Gap
raw_backtest_levels = (BACKTEST_CEILING - BACKTEST_FLOOR) / BACKTEST_GAP
if not np.isclose(raw_backtest_levels, round(raw_backtest_levels)):
    raise ValueError('(BACKTEST_CEILING - BACKTEST_FLOOR) must be exactly divisible by BACKTEST_GAP.')

NUMBER_OF_GRIDS = int(round(raw_backtest_levels))
CAPITAL_PER_LEVEL = BACKTEST_CAPITAL / NUMBER_OF_GRIDS

print('===== V0 Backtest Grid Configuration =====')
print(f'Historical Low     : {historical_low:,.2f} USDT')
print(f'Historical High    : {historical_high:,.2f} USDT')
print(f'Grid Floor         : {BACKTEST_FLOOR:,.2f} USDT')
print(f'Grid Ceiling       : {BACKTEST_CEILING:,.2f} USDT')
print(f'Grid Gap           : {BACKTEST_GAP:,.2f} USDT')
print(f'Number of Grids    : {NUMBER_OF_GRIDS}')
print(f'Capital            : {BACKTEST_CAPITAL:,.2f} USDT')
print(f'Capital / Grid     : {CAPITAL_PER_LEVEL:,.2f} USDT')
print(f'Buy Fee            : {BUY_FEE:.3%}')
print(f'Sell Fee           : {SELL_FEE:.3%}')

In [ ]:
# Build historical backtest grid with the same calculation logic as Excel
df_grid_backtest = build_excel_grid_table(
    capital=BACKTEST_CAPITAL,
    ceiling=BACKTEST_CEILING,
    floor=BACKTEST_FLOOR,
    gap=BACKTEST_GAP,
    buy_fee=BUY_FEE,
    sell_fee=SELL_FEE
)

assert len(df_grid_backtest) == NUMBER_OF_GRIDS
df_grid_backtest.head()

## 6. Backtest Engine — 1 Minute

The execution engine uses the **Excel grid calculation without changing the strategy formulas**.

V0 execution assumptions:
- Initial portfolio is 100% USDT and 0 BTC.
- Each grid interval can hold at most one open position.
- A BUY occurs only when price crosses **downward** through a buy level.
- A SELL occurs at the paired grid level one step above the buy price.
- Existing sell orders fill when the candle High reaches the sell target.
- Multiple grid levels can fill within one 1-minute candle.
- A position opened in a candle cannot close in that same candle.
- A grid sold in a candle cannot be bought again in that same candle.
- Sell proceeds from the current candle are not reused for BUY orders until the next candle.
- BUY fee is deducted from BTC received, exactly as in Excel.
- SELL fee is deducted from USDT proceeds, exactly as in Excel.
- Candle High is not used to infer a new downward BUY crossing, avoiding an unknown intrabar path assumption.

In [ ]:
def run_grid_backtest(df_price, grid_table, initial_capital):
    """Run the fixed arithmetic-grid strategy on 1-minute OHLC data."""
    required = {'open_time', 'open', 'high', 'low', 'close'}
    missing = required.difference(df_price.columns)
    if missing:
        raise ValueError(f'Missing price columns: {sorted(missing)}')
    if len(df_price) == 0:
        raise ValueError('df_price is empty.')
    if initial_capital <= 0:
        raise ValueError('initial_capital must be greater than 0.')

    data = df_price.sort_values('open_time').reset_index(drop=True)
    grid = grid_table.sort_values('buy_price').reset_index(drop=True).copy()
    if len(grid) == 0:
        raise ValueError('grid_table is empty.')

    buy_prices = grid['buy_price'].to_numpy(dtype=float)
    sell_prices = grid['sell_price'].to_numpy(dtype=float)
    capital_per_level = grid['capital_per_level'].to_numpy(dtype=float)
    base_amount = grid['base_amount'].to_numpy(dtype=float)
    buy_fee_base = grid['buy_fee_base'].to_numpy(dtype=float)
    sell_fee_quote = grid['sell_fee_quote'].to_numpy(dtype=float)
    net_sell = grid['net_sell'].to_numpy(dtype=float)
    cycle_profit = grid['profit'].to_numpy(dtype=float)
    excel_level = grid['level'].to_numpy(dtype=int)

    if len(grid) > 1 and not np.allclose(np.diff(buy_prices), np.diff(buy_prices)[0]):
        raise ValueError('V0 engine expects an arithmetic grid with a constant gap.')

    holding = np.zeros(len(grid), dtype=bool)
    buy_time = [None] * len(grid)
    sell_heap = []

    cash = float(initial_capital)
    open_btc = 0.0
    realized_profit = 0.0
    total_buy_fee_btc = 0.0
    total_buy_fee_usdt_equiv = 0.0
    total_sell_fee_usdt = 0.0
    completed_cycles = 0

    trade_events = []
    completed_trades = []

    n_rows = len(data)
    equity_values = np.empty(n_rows, dtype=float)
    cash_values = np.empty(n_rows, dtype=float)
    btc_values = np.empty(n_rows, dtype=float)
    buy_price_list = buy_prices.tolist()
    prev_close = None

    for i, row in enumerate(data.itertuples(index=False)):
        timestamp = row.open_time
        open_price = float(row.open)
        high_price = float(row.high)
        low_price = float(row.low)
        close_price = float(row.close)

        cash_at_candle_start = cash
        sold_this_candle = set()

        # 1) Existing SELL orders
        while sell_heap and sell_heap[0][0] <= high_price:
            _, k = heapq.heappop(sell_heap)
            if not holding[k]:
                continue

            holding[k] = False
            cash += net_sell[k]
            open_btc -= base_amount[k]
            if abs(open_btc) < 1e-12:
                open_btc = 0.0

            realized_profit += cycle_profit[k]
            total_sell_fee_usdt += sell_fee_quote[k]
            completed_cycles += 1
            sold_this_candle.add(k)

            completed_trades.append({
                'grid_level': int(excel_level[k]),
                'buy_time': buy_time[k],
                'sell_time': timestamp,
                'buy_price': buy_prices[k],
                'sell_price': sell_prices[k],
                'quote_cost': capital_per_level[k],
                'base_amount': base_amount[k],
                'net_sell': net_sell[k],
                'profit': cycle_profit[k]
            })

            trade_events.append({
                'time': timestamp, 'side': 'SELL',
                'grid_level': int(excel_level[k]),
                'price': sell_prices[k],
                'base_amount': base_amount[k],
                'quote_amount': net_sell[k],
                'fee_base': 0.0,
                'fee_quote': sell_fee_quote[k],
                'realized_profit': cycle_profit[k],
                'cash_after': cash
            })
            buy_time[k] = None

        # 2) Downward BUY crossings
        # Same-candle SELL proceeds are not available for BUYs.
        buy_budget = cash_at_candle_start
        down_start = open_price if prev_close is None else max(prev_close, open_price)

        if low_price < down_start:
            # Candidate levels: low_price <= buy_price < down_start
            first_idx = bisect.bisect_left(buy_price_list, low_price)
            stop_idx = bisect.bisect_left(buy_price_list, down_start)

            # Higher levels are crossed first during a downward move.
            for k in range(stop_idx - 1, first_idx - 1, -1):
                if holding[k] or k in sold_this_candle:
                    continue

                cost = capital_per_level[k]
                if buy_budget + 1e-12 < cost:
                    break

                holding[k] = True
                buy_time[k] = timestamp
                buy_budget -= cost
                cash -= cost
                open_btc += base_amount[k]

                total_buy_fee_btc += buy_fee_base[k]
                total_buy_fee_usdt_equiv += buy_fee_base[k] * buy_prices[k]
                heapq.heappush(sell_heap, (sell_prices[k], k))

                trade_events.append({
                    'time': timestamp, 'side': 'BUY',
                    'grid_level': int(excel_level[k]),
                    'price': buy_prices[k],
                    'base_amount': base_amount[k],
                    'quote_amount': cost,
                    'fee_base': buy_fee_base[k],
                    'fee_quote': 0.0,
                    'realized_profit': 0.0,
                    'cash_after': cash
                })

        # 3) Mark to market at candle Close
        equity_values[i] = cash + open_btc * close_price
        cash_values[i] = cash
        btc_values[i] = open_btc
        prev_close = close_price

    equity_curve = pd.DataFrame({
        'open_time': data['open_time'].to_numpy(),
        'close': data['close'].to_numpy(dtype=float),
        'cash': cash_values,
        'btc': btc_values,
        'equity': equity_values
    })

    running_peak = np.maximum.accumulate(equity_values)
    drawdown = equity_values / running_peak - 1.0
    equity_curve['drawdown'] = drawdown

    max_drawdown = float(drawdown.min())
    final_equity = float(equity_values[-1])
    net_return = final_equity / initial_capital - 1.0

    elapsed_days = (data['open_time'].iloc[-1] - data['open_time'].iloc[0]).total_seconds() / 86400.0
    annualized_return = np.nan
    if elapsed_days > 0 and final_equity > 0:
        annualized_log_growth = np.log(final_equity / initial_capital) * (365.25 / elapsed_days)
        if annualized_log_growth < 700:
            annualized_return = float(np.expm1(annualized_log_growth))

    calmar = np.nan
    if max_drawdown < 0 and np.isfinite(annualized_return):
        calmar = float(annualized_return / abs(max_drawdown))

    summary = {
        'initial_capital': float(initial_capital),
        'final_equity': final_equity,
        'net_return': float(net_return),
        'annualized_return': annualized_return,
        'max_drawdown': max_drawdown,
        'calmar_ratio': calmar,
        'completed_cycles': int(completed_cycles),
        'open_positions': int(holding.sum()),
        'final_cash': float(cash),
        'final_btc': float(open_btc),
        'realized_profit': float(realized_profit),
        'unrealized_pnl': float(final_equity - initial_capital - realized_profit),
        'buy_fee_btc': float(total_buy_fee_btc),
        'buy_fee_usdt_equiv': float(total_buy_fee_usdt_equiv),
        'sell_fee_usdt': float(total_sell_fee_usdt),
        'total_fee_usdt_equiv': float(total_buy_fee_usdt_equiv + total_sell_fee_usdt)
    }

    return {
        'summary': summary,
        'trade_log': pd.DataFrame(trade_events),
        'completed_trades': pd.DataFrame(completed_trades),
        'equity_curve': equity_curve,
        'grid_state': grid.assign(holding=holding, buy_time=buy_time)
    }

In [ ]:
# Run V0 fixed-grid backtest on BTCUSDT 1-minute data
backtest_result = run_grid_backtest(
    df_price=df_1m,
    grid_table=df_grid_backtest,
    initial_capital=BACKTEST_CAPITAL
)

backtest_summary = backtest_result['summary']
df_trade_log = backtest_result['trade_log']
df_completed_trades = backtest_result['completed_trades']
df_equity_curve = backtest_result['equity_curve']
df_grid_state = backtest_result['grid_state']

print('Backtest completed.')

## 7. Trade Log & Performance

In [ ]:
def print_backtest_summary(summary):
    print('===== V0 Backtest Summary =====')
    print(f'Initial Capital     : {summary["initial_capital"]:,.2f} USDT')
    print(f'Final Equity        : {summary["final_equity"]:,.2f} USDT')
    print(f'Net Return          : {summary["net_return"]:.2%}')

    if np.isfinite(summary['annualized_return']):
        print(f'Annualized Return   : {summary["annualized_return"]:.2%}')
    else:
        print('Annualized Return   : N/A')

    print(f'Max Drawdown        : {summary["max_drawdown"]:.2%}')

    if np.isfinite(summary['calmar_ratio']):
        print(f'Calmar Ratio        : {summary["calmar_ratio"]:.3f}')
    else:
        print('Calmar Ratio        : N/A')

    print(f'Completed Cycles    : {summary["completed_cycles"]:,}')
    print(f'Open Positions      : {summary["open_positions"]:,}')
    print(f'Final Cash          : {summary["final_cash"]:,.2f} USDT')
    print(f'Final BTC           : {summary["final_btc"]:.8f} BTC')
    print(f'Realized Profit     : {summary["realized_profit"]:,.2f} USDT')
    print(f'Unrealized P&L      : {summary["unrealized_pnl"]:,.2f} USDT')
    print(f'Buy Fee             : {summary["buy_fee_btc"]:.8f} BTC')
    print(f'Buy Fee (USDT eq.)  : {summary["buy_fee_usdt_equiv"]:,.2f} USDT')
    print(f'Sell Fee            : {summary["sell_fee_usdt"]:,.2f} USDT')
    print(f'Total Fee (USDT eq.): {summary["total_fee_usdt_equiv"]:,.2f} USDT')

print_backtest_summary(backtest_summary)

In [ ]:
# Completed BUY -> SELL grid cycles
df_completed_trades.head(10)

In [ ]:
# Event-level trade log
df_trade_log.head(20)

In [ ]:
# Grid intervals still holding BTC at the end of backtest
df_grid_state.loc[df_grid_state['holding']].copy()

In [ ]:
# Daily equity curve for an easier visual check
df_equity_daily = (
    df_equity_curve
    .set_index('open_time')['equity']
    .resample('1D')
    .last()
    .dropna()
)

plt.figure(figsize=(14, 5))
plt.plot(df_equity_daily.index, df_equity_daily.values)
plt.title('BTC Spot Fixed Grid — Daily Portfolio Equity')
plt.xlabel('Date')
plt.ylabel('Equity (USDT)')
plt.grid(True, alpha=0.3)
plt.show()